<h1 style="text-align:center;">Song Success Prediction Model</h1>
<h2 style="text-align:center;">Data Wrangling</h2>

<h4 style="text-align:center;">by Cameron Hicks</h3>

# Introduction

The goal of this notebook is to import data from the Spotify Analytics Dataset and combine with data pulled from www.apicountries.com, then to prepare that data Exploratory Data Analysis, Pre-processing, and Modeling. This project aims to create a Machine Learning Model that will predict the success of a song prior to or in early release phases. The steps taken in this notebook are necessary for Data Wrangling and include importing data from multiple sources, inspecting, cleaning, handling missing and duplicate values, and standardizing the data. The conclusion of this notebook ends with a new data set titled "combined_df" that is ready for Exploratory Data Analysis.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import requests
import time
from urllib.parse import quote
from sklearn.preprocessing import MultiLabelBinarizer
from scipy.stats import chi2_contingency
from scipy.stats import pointbiserialr

# Data Wrangling

## Spotify Data

### Import Data

Spotify Analytics Dataset imported into a DataFrame titled "raw_data".

Data Source: https://www.kaggle.com/datasets/rohiteng/spotify-music-analytics-dataset-20152025?resource=download

In [2]:
raw_data = pd.read_csv('../Data/spotify_2015_2025_85k.csv')

### Inspect Spotify Data

Before making any changes to the data it needs to be inspected. In the cells below I fully inspect the data to become familiar with it.

In [3]:
raw_data.shape

(85000, 19)

We can see that there are 85k records and 19 features in the raw data frame

In [4]:
raw_data.dtypes

track_id             object
track_name           object
artist_name          object
album_name           object
release_date         object
genre                object
duration_ms           int64
popularity            int64
danceability        float64
energy              float64
key                   int64
loudness            float64
mode                  int64
instrumentalness    float64
tempo               float64
stream_count          int64
country              object
explicit              int64
label                object
dtype: object

In [5]:
# Print the first 5 rows of the data to see what the DataFrame and records look like
raw_data.head()

,track_id,track_name,artist_name,album_name,release_date,genre,duration_ms,popularity,danceability,energy,key,loudness,mode,instrumentalness,tempo,stream_count,country,explicit,label
0,TRK-BEBD53DA84E1,Agent every (0),Noah Rhodes,Beautiful instead,2016-04-01,Pop,234194,55,0.15,0.74,9,-32.22,0,0.436,73.12,13000,Brazil,0,Universal Music
1,TRK-6A32496762D7,Night respond,Jennifer Cole,Table,2022-04-15,Metal,375706,45,0.44,0.46,0,-14.02,0,0.223,157.74,1000,France,1,Island Records
2,TRK-47AA7523463E,Future choice whatever,Brandon Davis,Page southern,2016-02-23,Rock,289191,55,0.62,0.80,8,-48.26,1,0.584,71.03,1000,Germany,1,XL Recordings
3,TRK-25ADA22E3B06,Bad fall pick those,Corey Jones,Spring,2015-10-12,Pop,209484,51,0.78,0.98,1,-34.47,1,0.684,149.00,1000,France,0,Warner Music
4,TRK-9245F2AD996A,Husband,Mark Diaz,Great prove,2022-07-08,Indie,127435,39,0.74,0.18,10,-17.84,0,0.304,155.85,2000,United States,0,Independent


Column insights from the head of the DataFrame

- <strong>Track ID</strong> records show unique id numbers that would not be easily identified without a key. There is not a visable system to determining the track id at first glance.
- <strong>Track Name, Artist Name,</strong> and <strong>Album Name</strong> are strings with the relavent data
- <strong>Release date</strong> contains the year, month, and day of the release in one column. It should be converted to a datetime object.
- <strong>Duration</strong> is measured in milliseconds. This will likely prove useful for computer analysis, but it may be helpful to add a column that shows the duration in minutes:seconds:milliseconds. 
- <strong>Popularity</strong> is measured by a two digit number for each example in the head. This column is our primary target feature for the model and will need to be inspected further.
- <strong>Danceability</strong> and <strong>Energy</strong> are measured as a float between 0 and 1
- <strong>Energy</strong> is float values between 0-1 showing the energy of the song
- <strong>Key</strong> is a numeric value incidating the key of the song
- <strong>Loudness</strong> is consistently a negative float representing the loudness of the track in decibels.
- <strong>Mode</strong> is a boolean column representing if the songs key is major or minor (0 = minor 1 = major)
- <strong>Instrumentalness</strong> is a float value between 0-1
- <strong>Tempo</strong> is Beats Per Minute
- <strong>Stream count</strong> is the number of streams for the track. We can assume that number of streams has a direct correlation with popularity of a song, and becuase our target model is to predict popularity prior to or at the early relase stages, this column will likely be dropped or become a target feature for the model to predict.
- <strong>Country</strong> represents what Country the song performed well. In Data Exploration it will need to be explored further to determine if songs that performed well in multiple countries have multiple recoreds, or if this column actually shows the top performing country for the song.
- <strong>Explicit</strong> - boolean showing if the song is marked as explicit
- <strong>Label</strong> is the label that released the song

### Edits and Additions to Spotify Data

In [6]:
# Ensure the release date is a datetime object
raw_data['release_date'] = pd.to_datetime(raw_data['release_date'])

In [7]:
raw_data.dtypes

track_id                    object
track_name                  object
artist_name                 object
album_name                  object
release_date        datetime64[ns]
genre                       object
duration_ms                  int64
popularity                   int64
danceability               float64
energy                     float64
key                          int64
loudness                   float64
mode                         int64
instrumentalness           float64
tempo                      float64
stream_count                 int64
country                     object
explicit                     int64
label                       object
dtype: object

In [8]:
# Add a second duration column for EDA.
raw_data['duration_mm:ss:ms'] = pd.to_timedelta(raw_data['duration_ms'], unit='ms').dt.components.apply(
    lambda x: f"{x.minutes:02d}:{x.seconds:02d}:{x.milliseconds:02d}", axis=1)

In [9]:
raw_data.head()

,track_id,track_name,artist_name,album_name,release_date,genre,duration_ms,popularity,danceability,energy,key,loudness,mode,instrumentalness,tempo,stream_count,country,explicit,label,duration_mm:ss:ms
0,TRK-BEBD53DA84E1,Agent every (0),Noah Rhodes,Beautiful instead,2016-04-01,Pop,234194,55,0.15,0.74,9,-32.22,0,0.436,73.12,13000,Brazil,0,Universal Music,03:54:194
1,TRK-6A32496762D7,Night respond,Jennifer Cole,Table,2022-04-15,Metal,375706,45,0.44,0.46,0,-14.02,0,0.223,157.74,1000,France,1,Island Records,06:15:706
2,TRK-47AA7523463E,Future choice whatever,Brandon Davis,Page southern,2016-02-23,Rock,289191,55,0.62,0.80,8,-48.26,1,0.584,71.03,1000,Germany,1,XL Recordings,04:49:191
3,TRK-25ADA22E3B06,Bad fall pick those,Corey Jones,Spring,2015-10-12,Pop,209484,51,0.78,0.98,1,-34.47,1,0.684,149.00,1000,France,0,Warner Music,03:29:484
4,TRK-9245F2AD996A,Husband,Mark Diaz,Great prove,2022-07-08,Indie,127435,39,0.74,0.18,10,-17.84,0,0.304,155.85,2000,United States,0,Independent,02:07:435


Next we will see how many unique values each column contains

In [10]:
# How many unique values does each column have?

for col in raw_data.columns:
    print(f"{col}: {raw_data[col].nunique()}")

track_id: 85000
track_name: 68951
artist_name: 62391
album_name: 43170
release_date: 4018
genre: 12
duration_ms: 75084
popularity: 99
danceability: 95
energy: 98
key: 12
loudness: 5401
mode: 2
instrumentalness: 801
tempo: 13967
stream_count: 2250
country: 10
explicit: 2
label: 8
duration_mm:ss:ms: 75084


Notice while there are 85,000 unique track id's there are only 68,951 unique track names. This could mean that there are songs that share the same name, or it could mean that some songs have duplicate records. Most likely it is a combination of both. This means we will need to be careful and thorough when inspecting and dropping duplicates.

### Missing and Null Values - Spotify Data

In this next section we will handle missing and null values.

In [11]:
raw_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 85000 entries, 0 to 84999
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   track_id           85000 non-null  object        
 1   track_name         84979 non-null  object        
 2   artist_name        85000 non-null  object        
 3   album_name         84954 non-null  object        
 4   release_date       85000 non-null  datetime64[ns]
 5   genre              85000 non-null  object        
 6   duration_ms        85000 non-null  int64         
 7   popularity         85000 non-null  int64         
 8   danceability       85000 non-null  float64       
 9   energy             85000 non-null  float64       
 10  key                85000 non-null  int64         
 11  loudness           85000 non-null  float64       
 12  mode               85000 non-null  int64         
 13  instrumentalness   85000 non-null  float64       
 14  tempo 

Inspecting the info for the raw dataframe, we can see that there are some columns that do not have 85000 non-null values. 

In [12]:
raw_data.isnull().sum()

track_id              0
track_name           21
artist_name           0
album_name           46
release_date          0
genre                 0
duration_ms           0
popularity            0
danceability          0
energy                0
key                   0
loudness              0
mode                  0
instrumentalness      0
tempo                 0
stream_count          0
country               0
explicit              0
label                 0
duration_mm:ss:ms     0
dtype: int64

Track name contains 21 missing values.

In [13]:
raw_data[raw_data['track_name'].isna()].sort_values(by='artist_name')

,track_id,track_name,artist_name,album_name,release_date,genre,duration_ms,popularity,danceability,energy,key,loudness,mode,instrumentalness,tempo,stream_count,country,explicit,label,duration_mm:ss:ms
71667,TRK-25F6BEF5437F,NaN,Aaron Nixon,Sense,2018-03-11,Jazz,142662,54,0.24,0.99,10,-22.01,0,0.784,106.37,18000,Germany,0,Independent,02:22:662
47946,TRK-04F201CCB2A2,NaN,Abigail Thomas,Quality police,2024-09-02,Country,281128,65,0.50,0.74,2,-17.92,1,0.352,89.28,79000,Brazil,0,XL Recordings,04:41:128
13675,TRK-530AF3473134,NaN,Brenda Calderon,Offer despite,2019-02-27,Indie,159043,35,0.51,0.58,9,-48.51,0,0.415,67.81,1000,India,1,Sony Music,02:39:43
83574,TRK-A7D9098C8152,NaN,Daniel Cox,Director perhaps,2024-09-08,Indie,153590,59,0.73,0.60,2,-40.17,1,0.498,86.30,1000,Germany,0,Independent,02:33:590
49470,TRK-A14E1F1D05D9,NaN,Eric Flores,Option there,2023-07-06,Jazz,360475,31,0.67,0.47,10,-34.79,1,0.420,102.42,1000,Mexico,0,Sony Music,06:00:475
67131,TRK-EBA6A845A873,NaN,Erin Petersen,Page,2017-01-18,Folk,322652,31,0.83,0.23,10,-28.57,0,0.447,88.73,1000,Canada,0,Sony Music,05:22:652
80537,TRK-EA13F1E7FB64,NaN,Frank Wolfe,Reflect,2021-12-02,EDM,196142,52,0.85,0.89,8,-37.18,1,0.053,83.38,1000,United Kingdom,1,Warner Music,03:16:142
71882,TRK-FEABDC31C767,NaN,Gary Tanner,Throughout,2019-10-08,Hip-Hop,398762,65,0.65,0.20,3,-40.41,1,0.643,64.43,3000,United Kingdom,0,Columbia,06:38:762
21451,TRK-224386735616,NaN,Jesse Carpenter,Right,2021-06-11,Country,269462,50,0.32,0.72,8,-26.73,0,0.604,115.88,1000,United Kingdom,0,Warner Music,04:29:462
61767,TRK-6D055555336C,NaN,Jonathan Sloan,Get involve,2018-03-06,Metal,403909,53,0.44,0.42,7,-13.95,1,0.450,146.22,3000,Japan,0,XL Recordings,06:43:909


With 21 tracks that do not have values for track name, it is best to fill these values with the string "Unknown" to keep their data to train and test the model. It is possible that the clients utalize the final version of the model will have working titles or no track name for the new songs, and therefore may use the 'Unknown' or null value when using the model.

Before filling the missing values, are there any existing songs with the string 'Unknown' already?

In [14]:
unknown_tracks = raw_data[raw_data['track_name'].str.lower() == 'unknown']

print(f"There are {len(unknown_tracks)} tracks with the name Unknown")

There are 0 tracks with the name Unknown


With 0 existing track names as "Unknown", missing track names will be filled with the string "Unknwon"

In [15]:
raw_data['track_name'] = raw_data['track_name'].fillna('Unknown')

In [16]:
unknown_tracks_new = raw_data[raw_data['track_name'].str.lower() == 'unknown']

print(f"There are {len(unknown_tracks_new)} tracks with the name Unknown")

There are 21 tracks with the name Unknown


Album name contains 46 missing values

In [17]:
raw_data[raw_data['album_name'].isna()].sort_values(by='artist_name')

,track_id,track_name,artist_name,album_name,release_date,genre,duration_ms,popularity,danceability,energy,key,loudness,mode,instrumentalness,tempo,stream_count,country,explicit,label,duration_mm:ss:ms
21674,TRK-65DC223A2CFC,Popular west,Abigail Hall,NaN,2023-01-04,R&B,389171,48,0.34,0.94,0,-50.14,0,0.314,93.10,7000,France,0,Columbia,06:29:171
59848,TRK-48D99EDCE521,Fish will population,Amanda Harding,NaN,2024-01-19,Classical,307684,40,0.33,0.28,9,-34.30,0,0.658,176.70,1000,Germany,0,Columbia,05:07:684
65401,TRK-3C0B06599311,Suffer when and,Amanda Smith,NaN,2024-10-05,R&B,194777,78,0.23,0.66,0,-51.36,0,0.647,120.59,1298000,United States,1,Sony Music,03:14:777
13690,TRK-827D10EBAB35,News,Anna Villarreal,NaN,2023-08-08,Rock,149590,35,0.09,0.61,6,-47.29,1,0.128,128.09,1000,France,0,Universal Music,02:29:590
31598,TRK-073E69DAAF9D,Its part,Anne Burns,NaN,2015-06-13,Pop,133232,47,0.53,0.42,7,-47.02,1,0.677,133.69,1000,United Kingdom,1,Independent,02:13:232
59011,TRK-39F249470C99,Throw Democrat,Anthony Anderson,NaN,2017-04-04,Rock,328667,49,0.59,0.93,5,-45.25,0,0.228,170.44,1000,United Kingdom,1,Island Records,05:28:667
29079,TRK-54D6D4C1826C,Leader yeah,Anthony Daniels,NaN,2021-07-24,Jazz,156435,70,0.53,0.62,8,-6.54,1,0.450,69.56,19000,United Kingdom,0,Universal Music,02:36:435
37737,TRK-5AF9D6599F95,Study bill mind,Ashley Krueger,NaN,2021-08-20,Classical,232750,67,0.24,0.11,10,-25.58,1,0.744,73.22,1000,Mexico,0,Columbia,03:52:750
64361,TRK-8BF8D0EC5A26,Should around official official,Brandon White,NaN,2023-04-11,Metal,139769,64,0.92,0.67,8,-32.42,0,0.541,71.52,8000,United States,0,Island Records,02:19:769
47664,TRK-99BEEE17B44B,Determine model,Brent Ortiz,NaN,2016-02-03,EDM,225143,54,0.52,0.92,10,-25.22,1,0.486,90.04,1000,Canada,1,Warner Music,03:45:143


The album name may be missing for several reasons - possibly the song was released as a single, possibly the song is not associated with a specific album. With the developments in digital released music vs CD's, many artists are releaseing songs as they're complete rather than as a full album. Ideally our model will assist the clients with making determinations for the best strategy to relase songs, so we do not want to drop the album_name column completly at this stage.

However, the album name should not be simply filled with "Unknown" as we did for track_name becasue unlike track_name, there is no unique identifier for Album Name. If we filled with 'Unknown' or something similar, all 46 records with the same album name, the model may think these are all associated and create a fake relationship between them. To protect the integrity of our data these records with missing values for album_name will be dropped.

In [18]:
raw_data = raw_data.dropna(subset=['album_name'])

In [19]:
raw_data.shape

(84954, 20)

In [20]:
raw_data.isnull().sum()

track_id             0
track_name           0
artist_name          0
album_name           0
release_date         0
genre                0
duration_ms          0
popularity           0
danceability         0
energy               0
key                  0
loudness             0
mode                 0
instrumentalness     0
tempo                0
stream_count         0
country              0
explicit             0
label                0
duration_mm:ss:ms    0
dtype: int64

All missing values have been appropriately filled or dropped.

### Duplicate Values - Spotify Data

In [21]:
exact_duplicates = raw_data.duplicated().sum()
print(f"Therea are {exact_duplicates} identical records")

Therea are 0 identical records


In [22]:
trackid_duplicates = raw_data['track_id'].duplicated().sum()
print(f"Duplicated Track IDs: {trackid_duplicates}")

Duplicated Track IDs: 0


With 0 identical records, and 0 duplicated track_id in the dataset it appears there are no duplicate values. But what if a track was released on a single, then later re-released on an album and was assigned two different unique track id's?

In [23]:
song_duplicates = raw_data[raw_data.duplicated(subset=['track_name', 'artist_name'], keep=False)]
print(f"Same Song/Artist pairs: {len(song_duplicates)}")

Same Song/Artist pairs: 6


In [24]:
song_duplicates.sort_values(by=['track_name', 'artist_name'])

,track_id,track_name,artist_name,album_name,release_date,genre,duration_ms,popularity,danceability,energy,key,loudness,mode,instrumentalness,tempo,stream_count,country,explicit,label,duration_mm:ss:ms
23411,TRK-A82E8EFECB98,Much,Michael Johnson,Then,2017-04-09,Country,418279,38,0.06,0.09,0,-16.92,1,0.352,85.78,3000,Australia,0,Columbia,06:58:279
26515,TRK-92BA06C1E3B6,Much,Michael Johnson,Structure feeling,2025-10-22,Classical,113486,27,0.75,0.07,5,-4.85,0,0.436,79.05,1000,Brazil,0,Sony Music,01:53:486
7993,TRK-2C9D28974FB8,Nation,Brian Brown,Ten indicate,2018-07-04,Pop,344352,42,0.22,0.91,0,-13.68,1,0.059,177.17,3000,Canada,1,Columbia,05:44:352
70052,TRK-186996D361F3,Nation,Brian Brown,West whole,2019-05-17,Country,242090,65,0.34,0.43,5,-10.06,0,0.431,155.68,5000,Japan,1,Columbia,04:02:90
27950,TRK-137F1682414B,Very,Jennifer Torres,Data,2016-06-06,Pop,118473,40,0.89,0.11,9,-6.17,1,0.111,109.99,6000,United States,1,Independent,01:58:473
61960,TRK-042055F5DE76,Very,Jennifer Torres,Expert magazine,2016-04-24,R&B,381545,41,0.50,0.30,8,-38.20,0,0.448,181.85,6000,Canada,0,Island Records,06:21:545


We found that there are 3 songs that were re-released by their original artists, however inspecting the other attributes it is clear that these are re-releases of the original song in a different genre, energy, or other different aspect. We will keep these songs for not, but will flag them for consideration after exploratory data analysis. These records can be insightful to popularity based on how popular the re-relases are based on the chaged attributes

<strong>Based on these findings there are no identical duplicates to be dropped</strong>

With the spotify data fully wrangled, I will copy the DataFrame into a new DataFrame calleed "spotify_data" to make it clear when merging in the next steps

In [25]:
spotify_data = raw_data.copy()

# Spotify API

In [27]:
!pip install spotipy

In [29]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

import requests
import base64

In [30]:
client_id = '677d7ab51f524d6aa7ee439ec529641d'
client_secret = 'cb265d13dda147baaf259b0759b87689'

In [39]:
auth_str = f"{client_id}:{client_secret}"
auth_bytes = auth_str.encode("utf-8")
auth_base64 = base64.b64encode(auth_bytes).decode("utf-8")

url = "https://accounts.spotify.com/api/token"

headers = {
    "Authorization": f"Basic {auth_base64}",
    "Content-Type": "application/x-www-form-urlencoded"
}

data = {"grant_type": "client_credentials"}

response = requests.post(url, headers=headers, data=data)

print("Status Code:", response.status_code)
print("Response Text:", response.text)

if response.status_code == 200:
    print("Authentication Successful")
    token = response.json()["access_token"]
else:
    print("Authentication Failed")

Status Code: 200
Response Text: {"access_token":"BQDKvYI0wcDCFAwWCyIfVOTjOB73XPWMbMT3mLOkTkvCooRdx8-yq-kU1zXOoHrIkLsdVFQuFRMvfBbxHH1Ous_3GtAPaux2aAPMXbHSdFBq_NoQTfxSBZaCLkgRGUO1owtL53Adu8E","token_type":"Bearer","expires_in":3600}
Authentication Successful


In [38]:
headers = {
    "Authorization": f"Bearer {token}"
}

artist_id = "INSERT_VALID_ARTIST_ID"

artist_url = f"https://api.spotify.com/v1/artists/{artist_id}"
artist_response = requests.get(artist_url, headers=headers)

print("Status Code:", artist_response.status_code)
print("Response Text:", artist_response.text)

if artist_response.status_code == 200:
    print("Artist Fetch Successful")
else:
    print("Artist Fetch Failed")

Status Code: 400
Response Text: {"error": {"status": 400, "message": "Invalid base62 id" } }
❌ Artist Fetch Failed


In [35]:
sp_artist_data = pd.DataFrame(artist_data)

In [36]:
sp_artist_data.head()

,error
status,400
message,Invalid base62 id


## Country Data

### Importing Country Data

To add features and data for our model to train and test on, I will be adding country data to our data frame. The steps below are taken to import that country data.

In [26]:
unique_countries = spotify_data['country'].unique().tolist()

In [27]:
unique_countries

['Brazil',
 'France',
 'Germany',
 'United States',
 'Australia',
 'United Kingdom',
 'Japan',
 'Canada',
 'India',
 'Mexico']

United States and United Kingdom do not exactly match the names of the countries for the api call and must be changed

In [28]:
mapping = {
    'United States': 'United States of America',
    'United Kingdom': 'United Kingdom of Great Britain and Northern Ireland'
}

unique_countries = [mapping.get(country, country) for country in unique_countries]

In [29]:
unique_countries

['Brazil',
 'France',
 'Germany',
 'United States of America',
 'Australia',
 'United Kingdom of Great Britain and Northern Ireland',
 'Japan',
 'Canada',
 'India',
 'Mexico']

In [30]:
BASE_URL = "https://www.apicountries.com/name"

country_data = []

for country in unique_countries:
    encoded_country = quote(country)
    url = f"{BASE_URL}/{encoded_country}"

    retries = 3

    for attempt in range(retries):
        response = requests.get(url)

        if response.status_code == 200:
            data = response.json()

            if isinstance(data, list) and len(data) > 0:
                country_data.append(data[0])
            break

        elif response.status_code == 429:
            wait_time = 2 ** attempt
            time.sleep(wait_time)

        else:
            print(f"Failed to fetch {country}: {response.status_code}")
            break

    time.sleep(1.2)

countries_df = pd.json_normalize(country_data)

countries_df.head()

,name,topLevelDomain,alpha2Code,alpha3Code,callingCodes,capital,altSpellings,subregion,region,population,...,translations.pt,translations.nl,translations.hr,translations.fa,translations.de,translations.es,translations.fr,translations.ja,translations.it,translations.hu
0,Brazil,[.br],BR,BRA,[55],Brasília,"[BR, Brasil, Federative Republic of Brazil, Re...",South America,Americas,212559409,...,Brasil,Brazilië,Brazil,برزیل,Brasilien,Brasil,Brésil,ブラジル,Brasile,Brazília
1,France,[.fr],FR,FRA,[33],Paris,"[FR, French Republic, République française]",Western Europe,Europe,67391582,...,França,Frankrijk,Francuska,فرانسه,Frankreich,Francia,France,フランス,Francia,Franciaország
2,Germany,[.de],DE,DEU,[49],Berlin,"[DE, Federal Republic of Germany, Bundesrepubl...",Central Europe,Europe,83240525,...,Alemanha,Duitsland,Njemačka,آلمان,Deutschland,Alemania,Allemagne,ドイツ,Germania,Grúzia
3,United States of America,[.us],US,USA,[1],"Washington, D.C.","[US, USA, United States of America]",Northern America,Americas,329484123,...,Estados Unidos,Verenigde Staten,Sjedinjene Američke Države,ایالات متحده آمریکا,Vereinigte Staaten von Amerika,Estados Unidos,États-Unis,アメリカ合衆国,Stati Uniti D'America,Amerikai Egyesült Államok
4,Australia,[.au],AU,AUS,[61],Canberra,[AU],Australia and New Zealand,Oceania,25687041,...,Austrália,Australië,Australija,استرالیا,Australien,Australia,Australie,オーストラリア,Australia,Ausztrália


In [31]:
countries_df

,name,topLevelDomain,alpha2Code,alpha3Code,callingCodes,capital,altSpellings,subregion,region,population,...,translations.pt,translations.nl,translations.hr,translations.fa,translations.de,translations.es,translations.fr,translations.ja,translations.it,translations.hu
0,Brazil,[.br],BR,BRA,[55],Brasília,"[BR, Brasil, Federative Republic of Brazil, Re...",South America,Americas,212559409,...,Brasil,Brazilië,Brazil,برزیل,Brasilien,Brasil,Brésil,ブラジル,Brasile,Brazília
1,France,[.fr],FR,FRA,[33],Paris,"[FR, French Republic, République française]",Western Europe,Europe,67391582,...,França,Frankrijk,Francuska,فرانسه,Frankreich,Francia,France,フランス,Francia,Franciaország
2,Germany,[.de],DE,DEU,[49],Berlin,"[DE, Federal Republic of Germany, Bundesrepubl...",Central Europe,Europe,83240525,...,Alemanha,Duitsland,Njemačka,آلمان,Deutschland,Alemania,Allemagne,ドイツ,Germania,Grúzia
3,United States of America,[.us],US,USA,[1],"Washington, D.C.","[US, USA, United States of America]",Northern America,Americas,329484123,...,Estados Unidos,Verenigde Staten,Sjedinjene Američke Države,ایالات متحده آمریکا,Vereinigte Staaten von Amerika,Estados Unidos,États-Unis,アメリカ合衆国,Stati Uniti D'America,Amerikai Egyesült Államok
4,Australia,[.au],AU,AUS,[61],Canberra,[AU],Australia and New Zealand,Oceania,25687041,...,Austrália,Australië,Australija,استرالیا,Australien,Australia,Australie,オーストラリア,Australia,Ausztrália
5,United Kingdom of Great Britain and Northern I...,[.uk],GB,GBR,[44],London,"[GB, UK, Great Britain]",Northern Europe,Europe,67215293,...,Reino Unido,Verenigd Koninkrijk,Ujedinjeno Kraljevstvo,بریتانیای کبیر و ایرلند شمالی,Vereinigtes Königreich,Reino Unido,Royaume-Uni,イギリス,Regno Unito,Nagy-Britannia
6,Japan,[.jp],JP,JPN,[81],Tokyo,"[JP, Nippon, Nihon]",Eastern Asia,Asia,125836021,...,Japão,Japan,Japan,ژاپن,Japan,Japón,Japon,日本,Giappone,Japán
7,Canada,[.ca],CA,CAN,[1],Ottawa,[CA],Northern America,Americas,38005238,...,Canadá,Canada,Kanada,کانادا,Kanada,Canadá,Canada,カナダ,Canada,Kanada
8,British Indian Ocean Territory,[.io],IO,IOT,[246],Diego Garcia,[IO],Eastern Africa,Africa,3000,...,Território Britânico do Oceano Índico,Britse Gebieden in de Indische Oceaan,Britanski Indijskooceanski teritorij,قلمرو بریتانیا در اقیانوس هند,Britisches Territorium im Indischen Ozean,Territorio Británico del Océano Índico,Territoire britannique de l'océan Indien,イギリス領インド洋地域,Territorio britannico dell'oceano indiano,Brit Indiai-óceáni Terület
9,Mexico,[.mx],MX,MEX,[52],Mexico City,"[MX, Mexicanos, United Mexican States, Estados...",North America,Americas,128932753,...,México,Mexico,Meksiko,مکزیک,Mexiko,México,Mexique,メキシコ,Messico,Mexikó


In [32]:
#save the raw country data as a csv file

countries_df.to_csv('../Data/country_raw_data.csv', index=False)

### Cleaning Country Data

Some of these columns are clearly not needed for this project. Those columns will be dropped:
- topLevelDomain
- callingCodes
- altSpellings
- flag
- flags.svg
- flags.png
- all translation columns

In [41]:
country_mapping = {
    'United States of America': 'United States',
    'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
    'British Indian Ocean Territory': 'India'
}

countries_df['name'] = countries_df['name'].replace(country_mapping)

/var/folders/vn/_qrvqjjx1yb4mwjbsrg0tnj80000gn/T/ipykernel_6261/1287605994.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  countries_df['name'] = countries_df['name'].replace(country_mapping)


In [ ]:
drop_cols = [
    'topLevelDomain', 
    'callingCodes',
    'altSpellings',
    'flag',
    'flags.svg',
    'flags.png'
]

countries_df.drop(columns=drop_cols, inplace=True)

In [44]:
countries_df = countries_df.loc[
    :, ~countries_df.columns.str.startswith("translations.")]

To ensure consistent formatting the region and subregion columns will be normalized into all lower case strings and numeric columns will be converted to numeric.

In [45]:
# Normalize columns
countries_df["region"] = countries_df["region"].str.lower()
countries_df["subregion"] = countries_df["subregion"].str.lower()

In [46]:
# Convert numeric columns

numeric_cols = [
    "population",
    "area",
    "gini",
    "numericCode"
]

countries_df[numeric_cols] = countries_df[numeric_cols].apply(
    pd.to_numeric, errors="coerce"
)

In [47]:
countries_df[numeric_cols].describe()

,population,area,gini,numericCode
count,1.000000e+01,1.000000e+01,9.000000,10.000000
mean,1.078355e+08,3.940461e+06,37.800000,339.000000
std,9.888682e+07,4.389695e+06,7.425968,297.229055
min,3.000000e+03,6.000000e+01,31.900000,36.000000
25%,4.530775e+07,3.623180e+05,32.900000,95.500000
50%,7.531605e+07,1.302527e+06,34.400000,263.000000
75%,1.281586e+08,8.309831e+06,41.400000,461.000000
max,3.294841e+08,9.984670e+06,53.400000,840.000000


In [48]:
countries_df[numeric_cols].isna().sum().sort_values(ascending=False)

gini           1
population     0
area           0
numericCode    0
dtype: int64

In [49]:
countries_df[countries_df['gini'].isna()]

,name,alpha2Code,alpha3Code,capital,subregion,region,population,latlng,demonym,area,gini,timezones,borders,nativeName,numericCode,currencies,languages,regionalBlocs,cioc,independent
8,India,IO,IOT,Diego Garcia,eastern africa,africa,3000,"[-6, 71.5]",Indian,60,NaN,[UTC+06:00],NaN,British Indian Ocean Territory,86,"[{'code': 'USD', 'name': 'United States dollar...","[{'iso639_1': 'en', 'iso639_2': 'eng', 'name':...","[{'acronym': 'AU', 'name': 'African Union', 'o...",NaN,True


India has a null value for gini. This data could be very important for our model. Since the API is unable to provide the information it will be filled with the value reported by the World Bank. This data shows the most recent report for India was in 2022 and the Gini was 25.5. That value will be manually input to preserve the dataset. This data was obtained from https://data.worldbank.org/indicator/SI.POV.GINI.

In [50]:
countries_df.loc[countries_df['name'] == 'India', 'gini'] = 25.5

In [51]:
countries_df['gini'].loc[countries_df['name'] == 'India']

8    25.5
Name: gini, dtype: float64

In [52]:
countries_df[numeric_cols].isna().sum().sort_values(ascending=False)

population     0
area           0
gini           0
numericCode    0
dtype: int64

In [53]:
countries_df.dtypes

name              object
alpha2Code        object
alpha3Code        object
capital           object
subregion         object
region            object
population         int64
latlng            object
demonym           object
area               int64
gini             float64
timezones         object
borders           object
nativeName        object
numericCode        int64
currencies        object
languages         object
regionalBlocs     object
cioc              object
independent         bool
dtype: object

### Edits and Additions to Country Data

While closely inspecting the DataFrame, several columns contain values of lists or lists of dictionaries. Each of these need to be handled individually

<strong>Timezones</strong><br>
The timezone column contains lists of the timezones that the country contains. Since there is a column containing 'region' information, it is not needed to know the specific time zones in a country, but it could be useful to know the total number of timezones a particular country contains.

In [54]:
countries_df['num_timezones'] = countries_df['timezones'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

<strong>Borders</strong><br>
The borders column contains lists of all the country codes that border the country. Similar to timezones, the data of specific countries borderd is not needed due to the region column. Once again we'll count the number of bording countries and use that data.

In [55]:
countries_df['num_borders'] = countries_df['borders'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

<strong>Currencies</strong><br>
The currencies column contains a list of a dictionary for each country with the official currency. Each dictionary contains the currency code, name and symbol for the currency. The code is sufficient for our data, so we will extract each code.

In [56]:
countries_df['currency_code'] = countries_df['currencies'].apply(
    lambda x: x[0]['code'] if isinstance(x, list) else None
)

<strong>Languages</strong><br>
The languages column contains a list of dictionaries with language information. Languages can have a significant impact on song popularity, so we will split this data into two important items: 1- a count of the number of languages spoken. 2- create individual columns for each unique language and input a value of 0 if the language is not commonly spoken in the country, or 1 if it is.

In [57]:
countries_df['num_languages'] = countries_df['languages'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

In [58]:
countries_df['language_names'] = countries_df['languages'].apply(
    lambda langs: [l['name'] for l in langs] if isinstance(langs, list) else []
)

In [59]:
mlb = MultiLabelBinarizer()

languages_encoded = pd.DataFrame(
    mlb.fit_transform(countries_df['language_names']),
    columns=[f"lang_{l.lower()}" for l in mlb.classes_],
    index=countries_df.index
)

In [60]:
countries_df = pd.concat([countries_df, languages_encoded], axis=1)

<strong>regionalBlocs</strong><br>
The regionalBlocs column contains a list of dictionaries that show what Associations/Unions the country is a part of. For this data we will create individual columns for each unique bloc and input a value of 0 if the country is not a member and a 1 if they are.

In [61]:
countries_df["regional_blocs"] = countries_df["regionalBlocs"].apply(
    lambda blocs: [b["acronym"] for b in blocs] if isinstance(blocs, list) else []
)

In [62]:
mlb = MultiLabelBinarizer()

blocs_encoded = pd.DataFrame(
    mlb.fit_transform(countries_df["regional_blocs"]),
    columns=[f"bloc_{b}" for b in mlb.classes_],
    index=countries_df.index
)

countries_df = pd.concat([countries_df, blocs_encoded], axis=1)

In [63]:
countries_df.columns.tolist()

['name',
 'alpha2Code',
 'alpha3Code',
 'capital',
 'subregion',
 'region',
 'population',
 'latlng',
 'demonym',
 'area',
 'gini',
 'timezones',
 'borders',
 'nativeName',
 'numericCode',
 'currencies',
 'languages',
 'regionalBlocs',
 'cioc',
 'independent',
 'num_timezones',
 'num_borders',
 'currency_code',
 'num_languages',
 'language_names',
 'lang_english',
 'lang_french',
 'lang_german',
 'lang_japanese',
 'lang_portuguese',
 'lang_spanish',
 'regional_blocs',
 'bloc_AU',
 'bloc_EU',
 'bloc_NAFTA',
 'bloc_PA',
 'bloc_USAN']

Now that this data has been cleaned and transformed these columns can be dropped:

- <strong>alpha3Code</strong> - duplicate information for alpha2Code
- <strong>timezones</strong> - replaced by num_timezones
- <strong>borders</strong> - repalced by num_boarders
- <strong>nativeName</strong> - duplicate information for name
- <strong>numericCode</strong> - duplicate information for name
- <strong>currencies</strong> - replaced by currency_code
- <strong>languages</strong> - replaced by num_languages and individual lang columns
- <strong>regionalBlocs</strong> - replaced by individual bloc columns
- <strong>language_names</strong> - created as a step to create individual lang columns
- <strong>regional_blocs</strong> - created as a step to create individual bloc columns

### Additional Cleaning of Country Data

In [64]:
drop_cols = [
    'alpha3Code',
 'timezones',
 'borders',
 'nativeName',
 'numericCode',
 'currencies',
 'languages',
 'regionalBlocs',
 'language_names',
 'regional_blocs'
]

countries_df.drop(columns=drop_cols, inplace=True)

In [65]:
countries_df

,name,alpha2Code,capital,subregion,region,population,latlng,demonym,area,gini,...,lang_french,lang_german,lang_japanese,lang_portuguese,lang_spanish,bloc_AU,bloc_EU,bloc_NAFTA,bloc_PA,bloc_USAN
0,Brazil,BR,Brasília,south america,americas,212559409,"[-10, -55]",Brazilian,8515767,53.4,...,0,0,0,1,0,0,0,0,0,1
1,France,FR,Paris,western europe,europe,67391582,"[46, 2]",French,640679,32.4,...,1,0,0,0,0,0,1,0,0,0
2,Germany,DE,Berlin,central europe,europe,83240525,"[51, 9]",German,357114,31.9,...,0,1,0,0,0,0,1,0,0,0
3,United States,US,"Washington, D.C.",northern america,americas,329484123,"[38, -97]",American,9629091,41.4,...,0,0,0,0,0,0,0,1,0,0
4,Australia,AU,Canberra,australia and new zealand,oceania,25687041,"[-27, 133]",Australian,7692024,34.4,...,0,0,0,0,0,0,0,0,0,0
5,United Kingdom,GB,London,northern europe,europe,67215293,"[54, -2]",British,242900,35.1,...,0,0,0,0,0,0,0,0,0,0
6,Japan,JP,Tokyo,eastern asia,asia,125836021,"[36, 138]",Japanese,377930,32.9,...,0,0,1,0,0,0,0,0,0,0
7,Canada,CA,Ottawa,northern america,americas,38005238,"[60, -95]",Canadian,9984670,33.3,...,1,0,0,0,0,0,0,1,0,0
8,India,IO,Diego Garcia,eastern africa,africa,3000,"[-6, 71.5]",Indian,60,25.5,...,0,0,0,0,0,1,0,0,0,0
9,Mexico,MX,Mexico City,north america,americas,128932753,"[23, -102]",Mexican,1964375,45.4,...,0,0,0,0,1,0,0,1,1,0


In [66]:
countries_df.isnull().sum()

name               0
alpha2Code         0
capital            0
subregion          0
region             0
population         0
latlng             0
demonym            0
area               0
gini               0
cioc               1
independent        0
num_timezones      0
num_borders        0
currency_code      0
num_languages      0
lang_english       0
lang_french        0
lang_german        0
lang_japanese      0
lang_portuguese    0
lang_spanish       0
bloc_AU            0
bloc_EU            0
bloc_NAFTA         0
bloc_PA            0
bloc_USAN          0
dtype: int64

<strong>cioc</strong> is the only column with a missing value. This data is the country code for the National Olympic Committee. This data will not be necessary for this model so the full column will be dropped.

In [67]:
countries_df.drop(columns='cioc', inplace=True)

In [68]:
countries_df.isnull().sum()

name               0
alpha2Code         0
capital            0
subregion          0
region             0
population         0
latlng             0
demonym            0
area               0
gini               0
independent        0
num_timezones      0
num_borders        0
currency_code      0
num_languages      0
lang_english       0
lang_french        0
lang_german        0
lang_japanese      0
lang_portuguese    0
lang_spanish       0
bloc_AU            0
bloc_EU            0
bloc_NAFTA         0
bloc_PA            0
bloc_USAN          0
dtype: int64

In [69]:
countries_df.dtypes

name                object
alpha2Code          object
capital             object
subregion           object
region              object
population           int64
latlng              object
demonym             object
area                 int64
gini               float64
independent           bool
num_timezones        int64
num_borders          int64
currency_code       object
num_languages        int64
lang_english         int64
lang_french          int64
lang_german          int64
lang_japanese        int64
lang_portuguese      int64
lang_spanish         int64
bloc_AU              int64
bloc_EU              int64
bloc_NAFTA           int64
bloc_PA              int64
bloc_USAN            int64
dtype: object

To be clear in the EDA step, the binary columns should be converted to boolean. 

In [70]:
binary_cols = [
    c for c in countries_df.columns
    if countries_df[c].isin([0, 1]).all()
]

countries_df[binary_cols] = countries_df[binary_cols].astype(bool)

In [71]:
countries_df.dtypes

name                object
alpha2Code          object
capital             object
subregion           object
region              object
population           int64
latlng              object
demonym             object
area                 int64
gini               float64
independent           bool
num_timezones        int64
num_borders          int64
currency_code       object
num_languages        int64
lang_english          bool
lang_french           bool
lang_german           bool
lang_japanese         bool
lang_portuguese       bool
lang_spanish          bool
bloc_AU               bool
bloc_EU               bool
bloc_NAFTA            bool
bloc_PA               bool
bloc_USAN             bool
dtype: object

In [72]:
countries_df.shape

(10, 26)

## Merge DataFrames

The Spotify DataFrame and Country DataFrame are now individually wrangled and cleaned, they are ready to be merged into one DataFrame. They will be merged on the Country column of 

### Prepare Merge

In [73]:
spotify_data['country'].unique().tolist()

['Brazil',
 'France',
 'Germany',
 'United States',
 'Australia',
 'United Kingdom',
 'Japan',
 'Canada',
 'India',
 'Mexico']

In [74]:
countries_df['name'].unique().tolist()

['Brazil',
 'France',
 'Germany',
 'United States',
 'Australia',
 'United Kingdom',
 'Japan',
 'Canada',
 'India',
 'Mexico']

In [75]:
countries_df.rename(columns={'name': 'country_name'}, inplace=True)

In [76]:
spotify_data.rename(columns={'country': 'country_name'}, inplace=True)

In [77]:
spotify_data.columns.tolist()

['track_id',
 'track_name',
 'artist_name',
 'album_name',
 'release_date',
 'genre',
 'duration_ms',
 'popularity',
 'danceability',
 'energy',
 'key',
 'loudness',
 'mode',
 'instrumentalness',
 'tempo',
 'stream_count',
 'country_name',
 'explicit',
 'label',
 'duration_mm:ss:ms']

In [78]:
countries_df.columns.tolist()

['country_name',
 'alpha2Code',
 'capital',
 'subregion',
 'region',
 'population',
 'latlng',
 'demonym',
 'area',
 'gini',
 'independent',
 'num_timezones',
 'num_borders',
 'currency_code',
 'num_languages',
 'lang_english',
 'lang_french',
 'lang_german',
 'lang_japanese',
 'lang_portuguese',
 'lang_spanish',
 'bloc_AU',
 'bloc_EU',
 'bloc_NAFTA',
 'bloc_PA',
 'bloc_USAN']

### Merge

In [79]:
combined_df = spotify_data.merge(
    countries_df,
    on='country_name',
    how='left'
)

### Inspect Merged DataFrame

In [80]:
combined_df.head()

,track_id,track_name,artist_name,album_name,release_date,genre,duration_ms,popularity,danceability,energy,...,lang_french,lang_german,lang_japanese,lang_portuguese,lang_spanish,bloc_AU,bloc_EU,bloc_NAFTA,bloc_PA,bloc_USAN
0,TRK-BEBD53DA84E1,Agent every (0),Noah Rhodes,Beautiful instead,2016-04-01,Pop,234194,55,0.15,0.74,...,False,False,False,True,False,False,False,False,False,True
1,TRK-6A32496762D7,Night respond,Jennifer Cole,Table,2022-04-15,Metal,375706,45,0.44,0.46,...,True,False,False,False,False,False,True,False,False,False
2,TRK-47AA7523463E,Future choice whatever,Brandon Davis,Page southern,2016-02-23,Rock,289191,55,0.62,0.80,...,False,True,False,False,False,False,True,False,False,False
3,TRK-25ADA22E3B06,Bad fall pick those,Corey Jones,Spring,2015-10-12,Pop,209484,51,0.78,0.98,...,True,False,False,False,False,False,True,False,False,False
4,TRK-9245F2AD996A,Husband,Mark Diaz,Great prove,2022-07-08,Indie,127435,39,0.74,0.18,...,False,False,False,False,False,False,False,True,False,False


In [81]:
combined_df.shape

(84954, 45)

In [82]:
combined_df.isnull().sum()

track_id             0
track_name           0
artist_name          0
album_name           0
release_date         0
genre                0
duration_ms          0
popularity           0
danceability         0
energy               0
key                  0
loudness             0
mode                 0
instrumentalness     0
tempo                0
stream_count         0
country_name         0
explicit             0
label                0
duration_mm:ss:ms    0
alpha2Code           0
capital              0
subregion            0
region               0
population           0
latlng               0
demonym              0
area                 0
gini                 0
independent          0
num_timezones        0
num_borders          0
currency_code        0
num_languages        0
lang_english         0
lang_french          0
lang_german          0
lang_japanese        0
lang_portuguese      0
lang_spanish         0
bloc_AU              0
bloc_EU              0
bloc_NAFTA           0
bloc_PA    

In [83]:
combined_df.dtypes

track_id                     object
track_name                   object
artist_name                  object
album_name                   object
release_date         datetime64[ns]
genre                        object
duration_ms                   int64
popularity                    int64
danceability                float64
energy                      float64
key                           int64
loudness                    float64
mode                          int64
instrumentalness            float64
tempo                       float64
stream_count                  int64
country_name                 object
explicit                      int64
label                        object
duration_mm:ss:ms            object
alpha2Code                   object
capital                      object
subregion                    object
region                       object
population                    int64
latlng                       object
demonym                      object
area                        

### Final Data Wrangling Edits

Two more columns need to be converted to categorical and boolean columns for the EDA process:
- <strong>mode</strong> - it will be more clear to see if the song is Major or Minor rather than a 0 or 1 in EDA
- <strong>explicit</strong> - converting to boolean will be more clear in EDA

In [84]:
combined_df['mode'] = combined_df['mode'].map({0: 'minor', 1: 'major'})

In [85]:
combined_df['explicit'] = combined_df['explicit'].astype(bool)

In [86]:
combined_df.dtypes

track_id                     object
track_name                   object
artist_name                  object
album_name                   object
release_date         datetime64[ns]
genre                        object
duration_ms                   int64
popularity                    int64
danceability                float64
energy                      float64
key                           int64
loudness                    float64
mode                         object
instrumentalness            float64
tempo                       float64
stream_count                  int64
country_name                 object
explicit                       bool
label                        object
duration_mm:ss:ms            object
alpha2Code                   object
capital                      object
subregion                    object
region                       object
population                    int64
latlng                       object
demonym                      object
area                        

In [87]:
combined_df['mode'].head()

0    minor
1    minor
2    major
3    major
4    minor
Name: mode, dtype: object

In [88]:
# Save combined_df to be loaded in EDA

combined_df.to_csv('../Data/DataWranglingComplete_Combined_DF.csv', index=False)

## Data Wrangling - Conclusion

In the Data Wrangling phase, Spotify track data and country-level contextual data were imported, cleaned, and merged into a single master dataset (combined_df) in preparation for exploratory analysis and modeling. Each dataset was cleaned independently prior to merging, with careful attention paid to data types, missing values, redundant features, and feature relevance to the project’s predictive goal. Derived features were created where appropriate to improve interpretability during EDA, while safeguards were put in place to prevent feature leakage during modeling.

Complex, nested country attributes were systematically transformed into numeric and boolean features to ensure usability, consistency, and interpretability. After cleaning, normalization, and feature engineering, the resulting dataset was structured, complete, and well-suited for exploratory analysis, enabling meaningful investigation of relationships between song characteristics, contextual factors, and popularity outcomes.